[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/jeffery/jeffery.ipynb)

# Jeffery Orbits

This example computes Jeffery orbits, the tumbling of rigid particles carried by a slow, sticky shear flow. A few grains of rice stirred into a jar of honey show it: the grains do not simply turn with the honey, they line up with the flow, linger, then flip end over end, again and again. A rigid particle cannot stretch with the fluid, only turn, and the fluid presses on it until it turns at the one rate that leaves no torque on it, found by Jeffery in 1922. A sphere turns with the fluid, a rod comes back to where it started on a closed orbit, and an ellipsoid with three different axes never quite repeats.

The flow's gradient and the body's shape are modelled as maps on offsets, the shape being the body's stretch of the unit sphere applied twice, and the body's spin as a bivector, whose turning map is `Point.commutator(spin)`. Jeffery's condition is one equation between maps, `shape(turning) + turning(shape) == gradient(shape) - shape(gradient.adjoint())`. Both sides are maps that turn, so their curls, `Point.reverse().commutator(map(Point)).contract(1, 2)`, carry them whole, and one solve gives the spin for a body in any orientation, with no principal frame.

In the index notation of suspension mechanics, in the particle's principal frame, the relative spin reads as $\omega_{ij} - \Omega_{ij} = \frac{a_i^2 - a_j^2}{a_i^2 + a_j^2}\,E_{ij}$, with $\Omega$ the vorticity and $E$ the strain rate of the flow.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D
from examples.util.animation import save_figure, show_animation
from examples.fluids.jeffery import render

ga = VGA3D
mv = NumpyContext(ga).multivector
# Offsets, and their velocities, are antivectors with intrinsic orientation.
Point = ga.gatype.antivector()
Bivector = ga.gatype.bivector()
Rotor = ga.gatype.rotor()
Stretch = ga.gatype((Point, Point))                        # Point <- Point: the unit sphere onto the body
Shape = ga.gatype((Point, Point))                          # Point <- Point: the body's quadric
Gradient = ga.gatype((Point, Point))                       # Point <- Point: velocity per offset

axes = mv.basis()                                          # [3] Plane
# The planes across the flow and across the gradient, and the vortex plane, the plane the fluid turns in.
x, y, z = axes

## 1. The flow and the bodies

In simple shear, honey between a fixed plate and a sliding one, the velocity along the flow grows with the height above the plane across the gradient, `y & Point`: the flow's gradient is a map on offsets. Its curl, the bivector whose turning map is the gradient less its adjoint, is twice the spin of the fluid. Each body is the unit sphere stretched along its own axes, and its shape is the stretch applied twice: the surface is where `offset.dot(shape.solve(offset)) == 1`. A rotor turns the shape like any map, on its output and its input.

In [ ]:
# The bodies, a sphere, a rod and an ellipsoid with three different axes: their shapes, their semi-axes in
# proportion, and their size, the radius of the ball of the same volume, alike for all so that none crowds out the
# flow around it; their semi-axes; and the rod's long semi-axis over its short ones. The flow: its velocity along
# the first axis, per offset along the second.
shapes = np.array([[1.0, 1.0, 1.0], [4.0, 1.0, 1.0], [4.0, 2.0, 1.0]])
radius = 1.3
semi_axes = radius * shapes / np.prod(shapes, axis=-1, keepdims=True) ** (1 / 3)
aspect = 4.0
shear_rate = 1.0

In [ ]:
def curl(gradient: Gradient) -> Bivector:
    """The bivector whose turning map is the gradient less its adjoint."""
    # The reciprocal of a unit antivector is its reverse, and the commutator of two directions is the plane they span.
    return Point.reverse().commutator(gradient(Point)).contract(1, 2)        # [...] Bivector


gradient = shear_rate * x.dual() * (y & Point)                                 # [] Point <- Point
fluid_spin = curl(gradient) / 2                                                # [] Bivector
stretches = (axes.dual() * semi_axes * (axes & Point)).sum(axis=-1)            # [bodies] Point <- Point


def placed(stretches: Stretch, rotors: Rotor) -> Shape:
    """The shape of each body, the stretch twice, turned by its rotor."""
    shape = stretches(stretches)
    return rotors >> shape(rotors << Point)

## 2. Jeffery's spin

Under a spin, a bivector, each offset of the body moves at `offset.commutator(spin)`, its turning map. Jeffery's spin, the one that leaves no torque on the body, satisfies one equation between maps, `shape(turning) + turning(shape) == gradient(shape) - shape(gradient.adjoint())`. Both sides are maps that turn, so their curls carry them whole: a map from the spin to a bivector on the left, a bivector on the right, and one solve. For a sphere the shape is a multiple of the identity, and the spin is the fluid's.

In [ ]:
def spin(shape: Shape, gradient: Gradient) -> Bivector:
    """The torque-free spin of a rigid ellipsoid of the given shape in a flow of the given gradient."""
    turning = Point.commutator(Bivector)                                      # Point <- (Point, Bivector)
    left = shape(turning) + turning(shape)                                    # [...] Point <- (Point, Bivector)
    right = gradient(shape) - shape(gradient.adjoint())                       # [...] Point <- Point
    # Both sides turn, so their curls carry them whole: a map on spins, and a bivector.
    return curl(left).solve(curl(right))                                      # [...] Bivector


sphere_spin = spin(placed(stretches[0], mv.rotor()), gradient)                 # [] Bivector: the fluid's

## 3. Tumbling

Each step turns every body by its spin, taken halfway through the step, and all bodies from all starts step together as one batch. On the sphere of directions, the sphere's axis goes around the normal of the vortex plane at a steady rate. The rod's axis lingers along the flow, flips, and comes back to where it started after Jeffery's period, $2\pi(r + 1/r)$ over the shear rate for a rod of aspect $r$, on a closed orbit. The ellipsoid with three different axes never quite closes its path.

Each panel is one body: the sphere, the rod and the ellipsoid with three axes. The grey arrows are the frame: the flow's direction $x$, its gradient $y$ and the normal of the vortex plane $z$; each colour is the path of the long axis from one start, both its ends drawn.

In [ ]:
# The angles of each body's long axis from the normal of the vortex plane at the start, in the plane of the flow and that
# normal; the time span in periods of the rod's orbit, and the steps per period.
tilts = np.array([0.15, 0.4, 0.7, 1.0, 1.3])
n_periods = 2
n_steps_per_period = 120

In [ ]:
def step(rotors: Rotor, dt: float) -> Rotor:
    """The bodies turned for a time `dt` by their spin at the midpoint of the step."""
    halfway = (spin(placed(stretches[:, None], rotors), gradient) * (-dt / 4)).exp() * rotors
    return (spin(placed(stretches[:, None], halfway), gradient) * (-dt / 2)).exp() * rotors


def tumble(rotors: Rotor, dt: float, n_steps: int):
    """The bodies' orientations, from the start and after each step."""
    yield rotors
    for _ in range(n_steps):
        rotors = step(rotors, dt)
        yield rotors


period = 2 * np.pi * (aspect + 1 / aspect) / shear_rate
starts = ((x ^ z) * (-(np.pi / 2 - tilts) / 2)).exp()[None].broadcast_to((len(semi_axes), len(tilts)))   # [bodies, starts] Rotor
rotors = stack(list(tumble(starts, period / n_steps_per_period, n_periods * n_steps_per_period)))     # [steps + 1, bodies, starts] Rotor
trails = rotors >> x.dual()                                                    # [steps + 1, bodies, starts] Point
save_figure(render.draw_orbits(trails), "jeffery_orbits");

The bodies tumble side by side from one start, the flow's velocity drawn behind them in grey and the paths of the tips of their long axes in orange. This cell takes about twenty seconds, half of it ray casting the bodies and half drawing the frames.

In [ ]:
# The start shown, the steps per frame, the heights at which the flow is drawn, all where it moves, and the duration of
# a frame.
shown = 2
n_steps_per_frame = 3
heights = y.dual() * np.array([-3.0, -2.0, -1.0, 1.0, 2.0, 3.0])                # [heights] Point
duration_ms = 60

In [ ]:
def frames(rotors: Rotor):
    """Per frame, every body's orientation from the start shown, and the tip of its long axis."""
    for index in range(0, rotors.shape[0], n_steps_per_frame):
        orientations = rotors[index, :, shown]                                 # [bodies] Rotor
        yield orientations, orientations >> (x.dual() * semi_axes[:, 0])


show_animation(render.animate_tumbling(stretches, heights, gradient(heights), frames(rotors)), "jeffery_tumbling", duration_ms)

In [ ]:
# checks
# A sphere spins at the fluid's own rate.
np.testing.assert_allclose((sphere_spin - fluid_spin).norm().to_array(), 0.0, atol=1e-14)
# The rod's axis is back where it started after one period, and keeps Jeffery's orbit constant; the
# tolerances test the step.
rod = trails[:, 1]                                                             # [steps + 1, starts] Point
np.testing.assert_allclose((rod[n_steps_per_period] - rod[0]).norm().to_array(), 0.0, atol=2e-2)
flowing, across, spinning = x & rod, y & rod, z & rod                           # [steps + 1, starts] Scalar each
constant = ((flowing.squared() + aspect**2 * across.squared()) / (aspect**2 * spinning.squared())).to_array()
np.testing.assert_allclose(constant / constant[0], 1.0, rtol=2e-2)